# 🔊 Datathon Template — AUDIO DATA (Audio / Sound / Speech Classification)

| # | Step | কি করব |
|---|------|--------|
| 0 | Colab setup & Config | sample rate, clip length, mel settings |
| 1 | Load metadata | CSV (`filename,label`) বা folder (`train/<class>/x.wav`) |
| 2 | EDA | class balance, duration, sample rate, channel, corrupted/duplicate, loudness, silence, **শুনে দেখা** |
| 3 | Visualization | waveform, spectrogram, **mel spectrogram**, MFCC, class-average mel |
| 4 | Preprocessing | mono, resample, trim silence, normalize, pad/crop to fixed length |
| 5 | Augmentation | gain, noise, time shift, pitch shift, time stretch, **SpecAugment**, mixup |
| 6 | ⚡ Baseline A | handcrafted features (MFCC, chroma, spectral …) → tabular model zoo |
| 7 | ⚡ Baseline B | pretrained audio model (AST / wav2vec2) embedding → LogReg |
| 8 | Main model | log-mel spectrogram → pretrained CNN (timm) — K-Fold, AMP, mixup |
| 9 | Evaluation | confusion matrix, report, ভুল clip শুনে/দেখে বোঝা |
| 10 | Inference | লম্বা clip → sliding window average + TTA → submission |
| 11 | Other audio tasks | ASR (Whisper, বাংলা), multi-label, sound event detection, regression |

> **মূল idea:** audio → **mel spectrogram (একটা ছবি)** → image model। BirdCLEF সহ বেশিরভাগ audio competition এভাবেই জেতা হয়।
> কোনো data না দিলে notebook নিজেই একটা **synthetic sound dataset** (tone / chirp / noise / beeps) বানিয়ে নেবে।

## ☁️ Google Colab Setup (সবার আগে এটা চালাও)

**GPU চালু করা:** `Runtime → Change runtime type → T4 GPU → Save` (image / transformer এর জন্য জরুরি)

**Data আনার ৪টা উপায় — নিচের `DATA_SOURCE` বদলাও:**
| `DATA_SOURCE` | কিভাবে |
|---|---|
| `"demo"` | কিছু লাগবে না — notebook নিজেই practice data বানাবে |
| `"upload"` | Cell চালালে **Choose Files** button আসবে → `train.csv`, `test.csv`, `sample_submission.csv` (বা একটা `.zip`) select করো |
| `"drive"` | আগে Google Drive এ data রাখো (যেমন `MyDrive/datathon/audio/`) → `DRIVE_DATA_DIR` ঠিক করো → permission দাও |
| `"kaggle"` | Kaggle → Settings → **Create New Token** → `kaggle.json` download → cell চালালে upload করো; competition page এ আগে **rules accept** করতে হবে |

সব data শেষে `data/audio` folder এ আসবে, `.zip` থাকলে auto unzip হবে — তাই নিচের CONFIG এ path বদলাতে হবে না
(শুধু file / column এর নাম মিলিয়ে নিও)।

> ⚠️ Colab বন্ধ হলে `/content` এর সব file মুছে যায় — submission আর model notebook এর শেষ cell দিয়ে download / Drive এ save করো।

In [ ]:
import sys, os, shutil, zipfile, subprocess, importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# ---------- 1) যে library Colab এ নেই সেগুলো install ----------
NEED = {"librosa": "librosa", "soundfile": "soundfile", "timm": "timm", "seaborn": "seaborn", "lightgbm": "lightgbm", "transformers": "transformers"}          # {import_name: pip_name}
missing = [pip for mod, pip in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)

# ---------- 2) Data ----------
DATA_SOURCE = "demo"                                  # "demo" | "upload" | "drive" | "kaggle"
DRIVE_DATA_DIR = "/content/drive/MyDrive/datathon/audio"
KAGGLE_COMPETITION = ""                               # e.g. "titanic"  (URL: kaggle.com/competitions/<এই নাম>)
KAGGLE_DATASET = ""                                   # competition না হয়ে dataset হলে: "owner/dataset-name"
LOCAL_DATA_DIR = Path("data/audio")

def unzip_all(folder):
    while zips := list(Path(folder).rglob("*.zip")):                       # zip এর ভেতরে zip থাকলেও (Kaggle এ common)
        for z in zips:
            print("Unzipping", z.name)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(z.parent)
            z.unlink()

if IN_COLAB and DATA_SOURCE != "demo":
    from google.colab import files, drive
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    if DATA_SOURCE == "upload":
        for name in files.upload():
            shutil.move(name, LOCAL_DATA_DIR / name)
    elif DATA_SOURCE == "drive":
        drive.mount("/content/drive")
        shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR, dirs_exist_ok=True)   # local copy → Drive থেকে পড়ার চেয়ে অনেক দ্রুত
    elif DATA_SOURCE == "kaggle":
        # নতুন token (KGAT_...) বা পুরনো kaggle.json — যেটা আছে সেটা দিয়েই চলবে
        import getpass
        if not os.environ.get("KAGGLE_API_TOKEN") and not Path("~/.kaggle/kaggle.json").expanduser().exists():
            tok = getpass.getpass("Kaggle API token paste করো (লুকানো থাকবে, KGAT_... দিয়ে শুরু) অথবা খালি রেখে Enter দিয়ে kaggle.json upload করো: ").strip()
            if tok:
                os.environ["KAGGLE_API_TOKEN"] = tok
            else:
                files.upload()
                Path("~/.kaggle").expanduser().mkdir(exist_ok=True)
                shutil.move("kaggle.json", Path("~/.kaggle/kaggle.json").expanduser())
                os.chmod(Path("~/.kaggle/kaggle.json").expanduser(), 0o600)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"], check=False)
        if KAGGLE_COMPETITION:
            subprocess.run(["kaggle", "competitions", "download", "-c", KAGGLE_COMPETITION, "-p", str(LOCAL_DATA_DIR)], check=True)
        else:
            subprocess.run(["kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", str(LOCAL_DATA_DIR)], check=True)
    unzip_all(LOCAL_DATA_DIR)

if LOCAL_DATA_DIR.exists():
    print(f"\n📁 {LOCAL_DATA_DIR} এর ভেতরে:")
    for p in sorted(LOCAL_DATA_DIR.iterdir())[:30]:
        print("  ", p.name, "/" if p.is_dir() else f"({p.stat().st_size / 1e6:.2f} MB)")
print("IN_COLAB =", IN_COLAB, "| DATA_SOURCE =", DATA_SOURCE)

## 0. Setup & Config

In [ ]:
import os, random, time, hashlib, importlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from scipy.io import wavfile
from IPython.display import Audio, display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn import metrics

def try_import(name):
    try:
        return importlib.import_module(name)
    except ImportError:
        print(f"⚠️  '{name}' নেই → pip install {name} (এই অংশ skip হবে)")
        return None

sf = try_import("soundfile")
timm = try_import("timm")
lgb = try_import("lightgbm")
transformers = try_import("transformers")

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

In [ ]:
# ======================= CONFIG =======================
DATA_DIR = Path("data/audio")
DATA_FORMAT = "csv"                       # "csv" → train.csv (filename,label) | "folder" → train/<class>/*.wav
TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_AUDIO_DIR = DATA_DIR / "train"
TEST_AUDIO_DIR = DATA_DIR / "test"
SUBMISSION_FILE = DATA_DIR / "sample_submission.csv"
OUTPUT_DIR = Path("outputs/audio")
FILE_COL, LABEL_COL = "filename", "label"
AUDIO_EXT = ""                            # CSV তে extension না থাকলে ".wav" / ".ogg" / ".mp3"

SR = 16000                                # সব audio এই sample rate এ resample (speech: 16k, music/bird: 32k)
CLIP_SECONDS = 2.0                        # model এ যাবে এই দৈর্ঘ্যের clip (data এর median duration দেখে ঠিক করো)
MAX_TRAIN_CLIPS = None                    # বড় dataset এ RAM বাঁচাতে (যেমন 3000) — class ratio রেখে sample; None → সব
MAX_LOAD_SECONDS = 30                     # প্রতিটা file এর প্রথম এত সেকেন্ড পর্যন্ত load (লম্বা recording এ RAM বাঁচায়)
TRIM_SILENCE = True
N_FFT, HOP, N_MELS, FMIN, FMAX = 1024, 256, 128, 20, None   # FMAX None → SR/2

MODEL_NAME = "resnet18"                   # timm: "efficientnet_b0", "tf_efficientnetv2_s", "convnext_tiny"
PRETRAINED = True
EPOCHS = 15
BATCH_SIZE = 32
LR = 1e-3
LABEL_SMOOTHING = 0.1
USE_MIXUP = True
PATIENCE = 4
N_FOLDS = 5
TRAIN_FOLDS = [0]                         # সময় থাকলে [0,1,2,3,4]
SEED = 42
NUM_WORKERS = 0 if os.name == "nt" else 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

RUN_PRETRAINED_EMB = DEVICE.type == "cuda"             # pretrained audio transformer embedding (CPU তে slow)
AUDIO_EMB_MODEL = "MIT/ast-finetuned-audioset-10-10-0.4593"   # environmental sound; speech: "facebook/wav2vec2-base"
# ======================================================

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLIP_LEN = int(SR * CLIP_SECONDS)
print("Device:", DEVICE, "| librosa", librosa.__version__, "| torch", torch.__version__)
if DEVICE.type != "cuda" and IN_COLAB:
    print("⚠️ GPU চালু নেই! Runtime → Change runtime type → T4 GPU")

### (Practice) Demo dataset generator
৪টা class: **tone** (স্থির সুর), **chirp** (সুর উপরে ওঠে), **noise** (filtered noise burst), **beeps** (বারবার ছোট beep)।
ইচ্ছা করে রাখা: ভিন্ন sample rate (16k/22.05k/44.1k), কিছু stereo, ভিন্ন দৈর্ঘ্য, শুরু-শেষে নীরবতা, background noise,
একটা **corrupted file** আর একটা **duplicate**।

In [ ]:
def make_demo_audio(root, n_train=400, n_test=100, seed=SEED):
    rng = np.random.default_rng(seed)
    classes = ["tone", "chirp", "noise", "beeps"]

    def synth(cls):
        sr = int(rng.choice([16000, 22050, 44100]))
        dur = rng.uniform(0.6, 3.0)
        t = np.arange(int(sr * dur)) / sr
        f0 = rng.uniform(250, 900)
        if cls == "tone":
            y = np.sin(2 * np.pi * f0 * t) + 0.3 * np.sin(2 * np.pi * 2 * f0 * t)
        elif cls == "chirp":
            f = f0 + (f0 * rng.uniform(1.5, 3)) * t / dur
            y = np.sin(2 * np.pi * np.cumsum(f) / sr)
        elif cls == "noise":
            y = np.convolve(rng.normal(size=len(t)), np.ones(int(rng.integers(3, 12))) / 6, mode="same")
            y *= np.hanning(len(y)) ** 0.3
        else:
            pulse = (np.sin(2 * np.pi * rng.uniform(4, 8) * t) > 0.3).astype(float)
            y = pulse * np.sin(2 * np.pi * f0 * 1.5 * t)
        y = y / (np.abs(y).max() + 1e-9) * rng.uniform(0.2, 0.9)
        y = y + rng.normal(0, rng.uniform(0.005, 0.08), len(y))                          # background noise
        pad = lambda n: np.zeros(int(sr * n))
        y = np.concatenate([pad(rng.uniform(0, .4)), y, pad(rng.uniform(0, .4))])        # silence
        if rng.random() < 0.15:
            y = np.stack([y, y * rng.uniform(.6, 1)], axis=1)                            # stereo
        return sr, (np.clip(y, -1, 1) * 32767).astype(np.int16)

    for sub in ["train", "test"]:
        (root / sub).mkdir(parents=True, exist_ok=True)
    rows = []
    for i in range(n_train):
        cls = str(rng.choice(classes, p=[.3, .3, .25, .15]))
        sr, y = synth(cls); name = f"train_{i:04d}.wav"
        wavfile.write(root / "train" / name, sr, y); rows.append((name, cls))
    (root / "train" / "train_9998.wav").write_bytes(b"not really audio")                   # corrupted
    rows.append(("train_9998.wav", "tone"))
    (root / "train" / "train_9999.wav").write_bytes((root / "train" / rows[0][0]).read_bytes())  # duplicate
    rows.append(("train_9999.wav", rows[0][1]))
    pd.DataFrame(rows, columns=["filename", "label"]).to_csv(root / "train.csv", index=False)
    names = []
    for i in range(n_test):
        sr, y = synth(str(rng.choice(classes))); name = f"test_{i:04d}.wav"
        wavfile.write(root / "test" / name, sr, y); names.append(name)
    pd.DataFrame({"filename": names, "label": classes[0]}).to_csv(root / "sample_submission.csv", index=False)

DEMO = not (TRAIN_CSV.exists() or (DATA_FORMAT == "folder" and TRAIN_AUDIO_DIR.exists()))
if DEMO:
    print("Data পাওয়া যায়নি → DEMO audio dataset বানানো হচ্ছে")
    make_demo_audio(DATA_DIR)
    DATA_FORMAT, EPOCHS = "csv", 6

## 1. Load metadata

In [ ]:
AUDIO_EXTS = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac", ".opus"}

if DATA_FORMAT == "csv":
    df = pd.read_csv(TRAIN_CSV)
    df["path"] = df[FILE_COL].astype(str).apply(lambda x: str(TRAIN_AUDIO_DIR / f"{x}{AUDIO_EXT}"))
else:
    rows = [(str(p), p.parent.name) for p in sorted(TRAIN_AUDIO_DIR.rglob("*")) if p.suffix.lower() in AUDIO_EXTS]
    df = pd.DataFrame(rows, columns=["path", LABEL_COL]); df[FILE_COL] = df["path"].apply(lambda p: Path(p).name)

sample_sub = pd.read_csv(SUBMISSION_FILE) if SUBMISSION_FILE.exists() else None
if sample_sub is not None and FILE_COL in sample_sub:
    test_df = sample_sub[[FILE_COL]].copy()
else:
    test_df = pd.DataFrame({FILE_COL: sorted(p.name for p in TEST_AUDIO_DIR.rglob("*") if p.suffix.lower() in AUDIO_EXTS)})
test_df["path"] = test_df[FILE_COL].astype(str).apply(lambda x: str(TEST_AUDIO_DIR / (x if Path(x).suffix else f"{x}{AUDIO_EXT}")))

exists = df["path"].apply(os.path.exists)
print(f"Train: {len(df)} | Test: {len(test_df)} | Missing train files: {(~exists).sum()}")
df = df[exists].reset_index(drop=True)
if MAX_TRAIN_CLIPS and len(df) > MAX_TRAIN_CLIPS:
    frac = MAX_TRAIN_CLIPS / len(df)
    df = pd.concat([g.sample(max(1, int(len(g) * frac)), random_state=SEED) for _, g in df.groupby(LABEL_COL)]).reset_index(drop=True)
    print(f"Sampled {len(df)} clips (MAX_TRAIN_CLIPS)")
display(df.head())

## 2. EDA
### 2.1 Audio metadata — duration, sample rate, channels, corrupted, duplicate

In [ ]:
def audio_meta(path):
    info = {"ok": True, "sr": None, "channels": None, "duration": None, "kb": os.path.getsize(path) / 1024,
            "md5": hashlib.md5(open(path, "rb").read()).hexdigest()}
    try:
        if sf is not None:
            i = sf.info(path); info.update(sr=i.samplerate, channels=i.channels, duration=i.duration)
        else:
            y, sr = librosa.load(path, sr=None, mono=False); info.update(sr=sr, channels=1 if y.ndim == 1 else y.shape[0],
                                                                         duration=y.shape[-1] / sr)
    except Exception:
        info["ok"] = False
    return info

meta = pd.DataFrame([audio_meta(p) for p in df["path"]], index=df.index)
df = df.join(meta)
print("Corrupted files:", (~df.ok).sum(), df.loc[~df.ok, FILE_COL].tolist()[:10])
dups = df[df.duplicated("md5", keep=False)]
print(f"Duplicate files: {len(dups)} | same audio different label: {(dups.groupby('md5')[LABEL_COL].nunique() > 1).sum()}")
print("Sample rates:", df.sr.value_counts().to_dict(), "| Channels:", df.channels.value_counts().to_dict())
df = df[df.ok].drop_duplicates("md5").reset_index(drop=True)          # corrupted + duplicate বাদ

LE = LabelEncoder().fit(df[LABEL_COL])
df["y"] = LE.transform(df[LABEL_COL])
CLASSES = list(LE.classes_); N_CLASSES = len(CLASSES)
display(df.groupby(LABEL_COL)["duration"].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
vc = df[LABEL_COL].value_counts()
sns.barplot(x=vc.index.astype(str), y=vc.values, ax=axes[0]); axes[0].set_title(f"Class balance (ratio {vc.max() / vc.min():.2f})")
sns.histplot(data=df, x="duration", hue=LABEL_COL, bins=30, element="step", ax=axes[1])
axes[1].axvline(CLIP_SECONDS, c="k", ls="--"); axes[1].set_title(f"Duration (s) — dashed = CLIP_SECONDS")
sns.countplot(data=df, x="sr", ax=axes[2]); axes[2].set_title("Sample rate")
plt.tight_layout(); plt.show()
print(f"Duration median={df.duration.median():.2f}s, p95={df.duration.quantile(.95):.2f}s → CLIP_SECONDS এর আশেপাশে রাখো")

### 2.2 Load + preprocessing
**mono → resample (SR) → trim silence → peak normalize**। RAM এ cache করি (data খুব বড় হলে on-the-fly load করো)।

In [ ]:
def load_audio(path, sr=SR, trim=TRIM_SILENCE):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=MAX_LOAD_SECONDS)
    if trim and len(y):
        y_t, _ = librosa.effects.trim(y, top_db=30)
        y = y_t if len(y_t) > sr * 0.1 else y
    return (y / (np.abs(y).max() + 1e-9)).astype(np.float32)

def fix_length(y, n=CLIP_LEN, mode="center"):
    """ছোট হলে zero-pad, বড় হলে crop (train এ random, অন্য সময় center)।"""
    if len(y) < n:
        left = (n - len(y)) // 2 if mode == "center" else random.randint(0, n - len(y))
        return np.pad(y, (left, n - len(y) - left))
    start = (len(y) - n) // 2 if mode == "center" else random.randint(0, len(y) - n)
    return y[start:start + n]

t0 = time.time()
WAVS = [load_audio(p) for p in df["path"]]
TEST_WAVS = [load_audio(p) for p in test_df["path"]]
print(f"Loaded {len(WAVS)} train + {len(TEST_WAVS)} test clips in {time.time() - t0:.1f}s | RAM ≈ {sum(w.nbytes for w in WAVS + TEST_WAVS) / 1e6:.0f} MB")

# Raw vs processed
raw, raw_sr = librosa.load(df["path"].iloc[0], sr=None, mono=False)
fig, axes = plt.subplots(1, 2, figsize=(16, 3))
axes[0].plot(np.arange(raw.shape[-1]) / raw_sr, raw.T if raw.ndim > 1 else raw, lw=.5); axes[0].set_title(f"Raw ({raw_sr} Hz, {'stereo' if raw.ndim > 1 else 'mono'})")
axes[1].plot(np.arange(len(WAVS[0])) / SR, WAVS[0], lw=.5, c="g"); axes[1].set_title(f"Processed (mono, {SR} Hz, trimmed, normalized)")
plt.tight_layout(); plt.show()

### 2.3 শুনে দেখো 🎧 (প্রতিটা class থেকে একটা)

In [ ]:
for c in CLASSES:
    i = df.index[df[LABEL_COL] == c][0]
    print(f"▶ {c}  ({df.loc[i, FILE_COL]}, {df.loc[i, 'duration']:.2f}s)")
    display(Audio(WAVS[i], rate=SR))

### 2.4 Waveform, Spectrogram, Mel-spectrogram, MFCC (প্রতিটা class)

In [ ]:
def logmel_np(y):
    m = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, fmin=FMIN, fmax=FMAX)
    return librosa.power_to_db(m, ref=np.max)

fig, axes = plt.subplots(4, N_CLASSES, figsize=(4.5 * N_CLASSES, 11))
axes = np.array(axes).reshape(4, N_CLASSES)
for j, c in enumerate(CLASSES):
    y = WAVS[df.index[df[LABEL_COL] == c][0]]
    axes[0, j].plot(np.arange(len(y)) / SR, y, lw=.5); axes[0, j].set_title(f"{c} — waveform")
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP)), ref=np.max)
    librosa.display.specshow(D, sr=SR, hop_length=HOP, x_axis="time", y_axis="hz", ax=axes[1, j]); axes[1, j].set_title("STFT spectrogram")
    librosa.display.specshow(logmel_np(y), sr=SR, hop_length=HOP, x_axis="time", y_axis="mel", fmax=FMAX or SR / 2, ax=axes[2, j]); axes[2, j].set_title("Log-mel spectrogram")
    librosa.display.specshow(librosa.feature.mfcc(y=y, sr=SR, n_mfcc=20), x_axis="time", ax=axes[3, j]); axes[3, j].set_title("MFCC")
plt.tight_layout(); plt.show()

In [ ]:
# Class-average mel (fixed length এ) + loudness/silence per class
avg = [np.mean([logmel_np(fix_length(WAVS[i])) for i in df.index[df.y == k][:60]], 0) for k in range(min(N_CLASSES, 8))]
fig, axes = plt.subplots(1, len(avg), figsize=(4 * len(avg), 3))
for ax, a, c in zip(np.atleast_1d(axes), avg, CLASSES):
    ax.imshow(a, origin="lower", aspect="auto", cmap="magma"); ax.set_title(f"mean mel — {c}")
plt.tight_layout(); plt.show()

df["rms_db"] = [20 * np.log10(np.sqrt(np.mean(w ** 2)) + 1e-9) for w in WAVS]
df["silence_pct"] = [np.mean(np.abs(w) < 0.02) * 100 for w in WAVS]
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
sns.boxplot(data=df, x=LABEL_COL, y="rms_db", ax=axes[0]); axes[0].set_title("Loudness (RMS dB) per class")
sns.boxplot(data=df, x=LABEL_COL, y="silence_pct", ax=axes[1]); axes[1].set_title("Silence % per class")
plt.tight_layout(); plt.show()

## 3. Augmentation
| Type | কোথায় | Note |
|---|---|---|
| Gain, noise, time shift | waveform (দ্রুত) | সবসময় নিরাপদ |
| Pitch shift, time stretch | waveform (slow) | speech/music এ সাবধানে — class এর অর্থ বদলে যেতে পারে |
| **SpecAugment** (time/freq mask) | spectrogram | সবচেয়ে কার্যকর |
| Mixup | batch | ছোট data তে overfit কমায় |

In [ ]:
def wave_augment(y):
    y = y * 10 ** (random.uniform(-6, 6) / 20)                                   # gain ±6 dB
    if random.random() < 0.5:
        snr = random.uniform(10, 30)
        y = y + np.random.randn(len(y)).astype(np.float32) * (np.sqrt(np.mean(y ** 2)) / 10 ** (snr / 20))   # noise at SNR
    if random.random() < 0.5:
        y = np.roll(y, random.randint(-len(y) // 4, len(y) // 4))               # time shift
    return y.astype(np.float32)

def spec_augment(x, n_time=2, n_freq=2, max_t=0.15, max_f=0.15):
    """x: (B, mel, frames) → random time & frequency band গুলো mask (in-place copy)।"""
    x = x.clone()
    B, Fm, T_ = x.shape
    for b in range(B):
        for _ in range(n_freq):
            f = random.randint(0, int(Fm * max_f)); f0 = random.randint(0, Fm - f); x[b, f0:f0 + f, :] = 0
        for _ in range(n_time):
            t = random.randint(0, int(T_ * max_t)); t0 = random.randint(0, T_ - t); x[b, :, t0:t0 + t] = 0
    return x

y0 = fix_length(WAVS[0])
demos = {"original": y0, "gain+noise+shift": wave_augment(y0),
         "pitch +3 semitone": librosa.effects.pitch_shift(y0, sr=SR, n_steps=3),
         "time stretch 1.2x": fix_length(librosa.effects.time_stretch(y0, rate=1.2))}
fig, axes = plt.subplots(1, len(demos) + 1, figsize=(4 * (len(demos) + 1), 3))
for ax, (k, v) in zip(axes, demos.items()):
    ax.imshow(logmel_np(v), origin="lower", aspect="auto", cmap="magma"); ax.set_title(k)
sa = spec_augment(torch.tensor(logmel_np(y0) - logmel_np(y0).min())[None])[0]
axes[-1].imshow(sa, origin="lower", aspect="auto", cmap="magma"); axes[-1].set_title("SpecAugment")
plt.tight_layout(); plt.show()

## 4. ⚡ Baseline A — Handcrafted features → tabular model zoo
প্রতিটা clip থেকে MFCC/chroma/spectral feature এর **mean ও std** → একটা ছোট table → classical ML।
দ্রুত, GPU লাগে না, interpretable। (এই feature table টা `01_tabular_template` এও চালাতে পারো।)

In [ ]:
def handcrafted(y, sr=SR):
    y = y if len(y) >= sr // 2 else np.pad(y, (0, sr // 2 - len(y)))
    blocks = {
        "mfcc": librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20),
        "chroma": librosa.feature.chroma_stft(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP),
        "contrast": librosa.feature.spectral_contrast(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP),
        "centroid": librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP),
        "bandwidth": librosa.feature.spectral_bandwidth(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP),
        "rolloff": librosa.feature.spectral_rolloff(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP),
        "flatness": librosa.feature.spectral_flatness(y=y, n_fft=N_FFT, hop_length=HOP),
        "zcr": librosa.feature.zero_crossing_rate(y, hop_length=HOP),
        "rms": librosa.feature.rms(y=y, hop_length=HOP),
    }
    blocks["dmfcc"] = librosa.feature.delta(blocks["mfcc"], width=min(9, blocks["mfcc"].shape[1] // 2 * 2 - 1))
    vals, names = [], []
    for k, M in blocks.items():
        for stat, fn in [("mean", np.mean), ("std", np.std)]:
            vals.append(fn(M, axis=1)); names += [f"{k}{i}_{stat}" for i in range(M.shape[0])]
    vals.append([len(y) / sr]); names.append("duration")
    return np.concatenate(vals).astype(np.float32), names

t0 = time.time()
XF = np.stack([handcrafted(w)[0] for w in WAVS]); FEAT_NAMES = handcrafted(WAVS[0])[1]
XF_test = np.stack([handcrafted(w)[0] for w in TEST_WAVS])
print(f"Feature matrix {XF.shape} in {time.time() - t0:.1f}s")

skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
zoo = {
    "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=1.0)),
    "SVM (RBF)": make_pipeline(StandardScaler(), SVC(C=5, probability=True, random_state=SEED)),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(7)),
    "RandomForest": RandomForestClassifier(400, n_jobs=-1, random_state=SEED),
    "ExtraTrees": ExtraTreesClassifier(400, n_jobs=-1, random_state=SEED),
    "HistGB": HistGradientBoostingClassifier(random_state=SEED),
}
if lgb: zoo["LightGBM"] = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, verbose=-1, random_state=SEED)
feat_rows = []
for name, clf in zoo.items():
    s = cross_validate(clf, XF, df.y, cv=skf, scoring=["accuracy", "f1_macro"])
    feat_rows.append({"model": name, "acc": s["test_accuracy"].mean(), "f1_macro": s["test_f1_macro"].mean()})
    print(f"{name:13s} acc={feat_rows[-1]['acc']:.4f}  f1={feat_rows[-1]['f1_macro']:.4f}")
display(pd.DataFrame(feat_rows).sort_values("f1_macro", ascending=False))

et = ExtraTreesClassifier(400, n_jobs=-1, random_state=SEED).fit(XF, df.y)
pd.Series(et.feature_importances_, FEAT_NAMES).sort_values().tail(20).plot.barh(figsize=(7, 6), title="Top handcrafted features"); plt.show()

## 5. ⚡ Baseline B — Pretrained audio transformer embeddings (GPU recommended)
- **AST** (Audio Spectrogram Transformer, AudioSet এ trained) → environmental sound, music, animal
- **wav2vec2 / HuBERT / WavLM** → speech (keyword, emotion, speaker, language)
- বাংলা speech: `"facebook/wav2vec2-xls-r-300m"` (multilingual)

In [ ]:
EMB_OOF = None
if RUN_PRETRAINED_EMB and transformers is not None:
    from transformers import AutoFeatureExtractor, AutoModel
    fe = AutoFeatureExtractor.from_pretrained(AUDIO_EMB_MODEL)
    emb_model = AutoModel.from_pretrained(AUDIO_EMB_MODEL).to(DEVICE).eval()
    fe_sr = getattr(fe, "sampling_rate", 16000)

    @torch.no_grad()
    def embed(wavs, bs=16):
        out = []
        for i in range(0, len(wavs), bs):
            batch = [librosa.resample(w, orig_sr=SR, target_sr=fe_sr) if fe_sr != SR else w for w in wavs[i:i + bs]]
            inp = fe(batch, sampling_rate=fe_sr, return_tensors="pt", padding=True)
            o = emb_model(**{k: v.to(DEVICE) for k, v in inp.items()})
            pooled = o.pooler_output if getattr(o, "pooler_output", None) is not None else o.last_hidden_state.mean(1)
            out.append(pooled.float().cpu().numpy())
        return np.concatenate(out)

    t0 = time.time()
    E = embed(WAVS); E_test = embed(TEST_WAVS)
    print(f"Embeddings {E.shape} in {time.time() - t0:.1f}s")
    emb_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5))
    s = cross_validate(emb_clf, E, df.y, cv=skf, scoring=["accuracy", "f1_macro"])
    print(f"{AUDIO_EMB_MODEL} + LogReg  acc={s['test_accuracy'].mean():.4f}  f1={s['test_f1_macro'].mean():.4f}")
    import gc
    del emb_model, fe; gc.collect()                                              # বড় model RAM/GPU থেকে সরাও
    if torch.cuda.is_available(): torch.cuda.empty_cache()
else:
    print("Pretrained embedding skipped (RUN_PRETRAINED_EMB=False / transformers নেই)। GPU তে True দাও।")

## 6. Main model — Log-mel spectrogram + pretrained CNN
Waveform → (GPU তে) STFT → mel filterbank → log → normalize → **SpecAugment** → CNN (1-channel image)।
Mel GPU তে batch এ হিসাব হয় বলে training দ্রুত।

In [ ]:
class LogMel(nn.Module):
    def __init__(self):
        super().__init__()
        fb = librosa.filters.mel(sr=SR, n_fft=N_FFT, n_mels=N_MELS, fmin=FMIN, fmax=FMAX)
        self.register_buffer("fb", torch.tensor(fb, dtype=torch.float32))
        self.register_buffer("win", torch.hann_window(N_FFT))
    def forward(self, wav):                                                        # (B, T) → (B, mel, frames)
        spec = torch.stft(wav, N_FFT, HOP, window=self.win, return_complex=True).abs() ** 2
        x = torch.log(torch.matmul(self.fb, spec) + 1e-6)
        return (x - x.mean((1, 2), keepdim=True)) / (x.std((1, 2), keepdim=True) + 1e-6)   # per-clip normalize

def build_backbone(num_classes, pretrained=PRETRAINED):
    if timm is not None:
        try:
            return timm.create_model(MODEL_NAME, pretrained=pretrained, num_classes=num_classes, in_chans=1)
        except Exception as e:
            print(f"⚠️ pretrained load failed ({type(e).__name__}) → random init")
            return timm.create_model(MODEL_NAME, pretrained=False, num_classes=num_classes, in_chans=1)
    m = torchvision.models.resnet18(weights=None)
    m.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False); m.fc = nn.Linear(512, num_classes)
    return m

class AudioCNN(nn.Module):
    def __init__(self, num_classes=N_CLASSES, pretrained=PRETRAINED):
        super().__init__()
        self.mel = LogMel(); self.backbone = build_backbone(num_classes, pretrained)
    def forward(self, wav, augment=False):
        with torch.autocast(device_type=DEVICE.type, enabled=False):
            x = self.mel(wav.float())
        if augment:
            x = spec_augment(x)
        return self.backbone(x.unsqueeze(1))

class WaveDS(Dataset):
    def __init__(self, wavs, labels=None, train=False):
        self.wavs, self.labels, self.train = wavs, labels, train
    def __len__(self):
        return len(self.wavs)
    def __getitem__(self, i):
        y = fix_length(self.wavs[i], mode="random" if self.train else "center")
        if self.train:
            y = wave_augment(y)
        x = torch.from_numpy(np.ascontiguousarray(y, dtype=np.float32))
        return x if self.labels is None else (x, torch.tensor(self.labels[i], dtype=torch.long))

def make_loader(wavs, labels=None, train=False):
    return DataLoader(WaveDS(wavs, labels, train), batch_size=BATCH_SIZE, shuffle=train, num_workers=NUM_WORKERS,
                      drop_last=train and len(wavs) > BATCH_SIZE, pin_memory=DEVICE.type == "cuda")

with torch.no_grad():
    _m = AudioCNN(pretrained=False)
    print("Mel input shape:", tuple(_m.mel(torch.zeros(2, CLIP_LEN)).shape), "→ logits", tuple(_m(torch.zeros(2, CLIP_LEN)).shape))
    del _m

In [ ]:
def train_one_epoch(model, loader, opt, sched, criterion, scaler):
    model.train(); tot, n = 0., 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            if USE_MIXUP and random.random() < 0.5:
                lam = np.random.beta(0.4, 0.4); idx = torch.randperm(x.size(0), device=DEVICE)
                out = model(lam * x + (1 - lam) * x[idx], augment=True)
                loss = lam * criterion(out, y) + (1 - lam) * criterion(out, y[idx])
            else:
                loss = criterion(model(x, augment=True), y)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item() * len(x); n += len(x)
    return tot / n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval(); tot, P, Y = 0., [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            out = model(x).float()
        tot += criterion(out, y).item() * len(x); P.append(out.softmax(1).cpu()); Y.append(y.cpu())
    P, Y = torch.cat(P).numpy(), torch.cat(Y).numpy()
    return tot / len(Y), P, Y

def fit_fold(fold, tr_idx, va_idx):
    seed_everything(SEED + fold)
    model = AudioCNN().to(DEVICE)
    tl = make_loader([WAVS[i] for i in tr_idx], df.y.values[tr_idx], train=True)
    vl = make_loader([WAVS[i] for i in va_idx], df.y.values[va_idx])
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(tl), pct_start=0.15)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    ckpt = OUTPUT_DIR / f"audio_{MODEL_NAME}_fold{fold}.pt"
    hist, best, best_P, bad = [], -1, None, 0
    for ep in range(EPOCHS):
        t0 = time.time()
        trl = train_one_epoch(model, tl, opt, sched, criterion, scaler)
        val, P, Y = evaluate(model, vl, criterion)
        f1 = metrics.f1_score(Y, P.argmax(1), average="macro"); acc = metrics.accuracy_score(Y, P.argmax(1))
        hist.append({"epoch": ep + 1, "train_loss": trl, "val_loss": val, "val_acc": acc, "val_f1": f1})
        flag = ""
        if f1 > best:
            best, best_P, bad, flag = f1, P, 0, " ✅ saved"; torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
        print(f"fold {fold} | ep {ep + 1:02d} | train {trl:.4f} | val {val:.4f} | acc {acc:.4f} | f1 {f1:.4f} | {time.time() - t0:.0f}s{flag}")
        if bad >= PATIENCE:
            print("Early stopping"); break
    return pd.DataFrame(hist), best_P, ckpt

folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(df, df.y))
HISTORY, CKPTS = {}, []
oof = np.full((len(df), N_CLASSES), np.nan)
for fold in TRAIN_FOLDS:
    tr_idx, va_idx = folds[fold]
    HISTORY[fold], P, ck = fit_fold(fold, tr_idx, va_idx)
    oof[va_idx] = P; CKPTS.append(ck)

## 7. Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for f, h in HISTORY.items():
    axes[0].plot(h.epoch, h.train_loss, "o-", label=f"train f{f}"); axes[0].plot(h.epoch, h.val_loss, "s--", label=f"val f{f}")
    axes[1].plot(h.epoch, h.val_acc, "o-", label=f"acc f{f}"); axes[1].plot(h.epoch, h.val_f1, "s--", label=f"f1 f{f}")
axes[0].legend(); axes[1].legend(); axes[0].set_title("Loss"); axes[1].set_title("Validation"); plt.show()

mask = ~np.isnan(oof).any(1)
y_true, y_pred = df.y.values[mask], oof[mask].argmax(1)
print(f"CNN OOF acc = {metrics.accuracy_score(y_true, y_pred):.4f} | macro-F1 = {metrics.f1_score(y_true, y_pred, average='macro'):.4f}")
print(metrics.classification_report(y_true, y_pred, labels=range(N_CLASSES), target_names=[str(c) for c in CLASSES], digits=4, zero_division=0))
fig, axes = plt.subplots(1, 2, figsize=(max(12, N_CLASSES), max(4.5, .45 * N_CLASSES)))
metrics.ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=range(N_CLASSES), display_labels=CLASSES, ax=axes[0], cmap="Blues", colorbar=False, xticks_rotation=45)
metrics.ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=range(N_CLASSES), display_labels=CLASSES, ax=axes[1], cmap="Greens", normalize="true", colorbar=False, xticks_rotation=45, values_format=".2f")
plt.tight_layout(); plt.show()

In [ ]:
# ভুল clip গুলো শুনে আর দেখে বোঝো (label noise? দুই class এর মিশ্রণ? silence বেশি?)
err = np.where(mask)[0][y_true != y_pred]
err = err[np.argsort(-oof[err].max(1))][:6]
if len(err):
    fig, axes = plt.subplots(1, len(err), figsize=(4 * len(err), 3))
    for ax, i in zip(np.atleast_1d(axes), err):
        ax.imshow(logmel_np(fix_length(WAVS[i])), origin="lower", aspect="auto", cmap="magma")
        ax.set_title(f"true={CLASSES[df.y[i]]}\npred={CLASSES[oof[i].argmax()]} ({oof[i].max():.2f})", fontsize=9)
    plt.tight_layout(); plt.show()
    for i in err[:3]:
        print(f"▶ {df.loc[i, FILE_COL]} true={CLASSES[df.y[i]]} pred={CLASSES[oof[i].argmax()]}"); display(Audio(WAVS[i], rate=SR))
else:
    print("কোনো ভুল নেই 🎉")

## 8. Inference — লম্বা clip এ sliding window + TTA → Submission
Clip `CLIP_SECONDS` এর চেয়ে লম্বা হলে ৫০% overlap এ টুকরো করে সব টুকরোর probability গড়।

In [ ]:
@torch.no_grad()
def predict_clips(model, wavs, hop_ratio=0.5, tta=True):
    model.eval(); out = []
    for w in wavs:
        if len(w) <= CLIP_LEN:
            chunks = [fix_length(w)]
        else:
            step = int(CLIP_LEN * hop_ratio)
            chunks = [w[s:s + CLIP_LEN] for s in range(0, len(w) - CLIP_LEN + 1, step)] + [w[-CLIP_LEN:]]
        x = torch.from_numpy(np.stack(chunks)).to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            p = model(x).float().softmax(1)
            if tta:                                                                 # TTA: ছোট time shift
                p = (p + model(torch.roll(x, CLIP_LEN // 10, dims=1)).float().softmax(1)) / 2
        out.append(p.mean(0).cpu().numpy())                                         # লম্বা clip: max(0) ও try করো
    return np.stack(out)

test_probs = np.zeros((len(TEST_WAVS), N_CLASSES))
for ck in CKPTS:
    m = AudioCNN(pretrained=False).to(DEVICE)
    m.load_state_dict(torch.load(ck, map_location=DEVICE))
    test_probs += predict_clips(m, TEST_WAVS) / len(CKPTS)

# Optional: handcrafted-feature model এর সাথে blend (দুটো ভিন্ন ধরনের model → ensemble লাভ)
BLEND_W = 0.2
feat_best = make_pipeline(StandardScaler(), SVC(C=5, probability=True, random_state=SEED)).fit(XF, df.y)
final_probs = (1 - BLEND_W) * test_probs + BLEND_W * feat_best.predict_proba(XF_test)

sub = pd.DataFrame({FILE_COL: test_df[FILE_COL], LABEL_COL: LE.inverse_transform(final_probs.argmax(1))})
if sample_sub is not None:
    sub.columns = sample_sub.columns[:2]; assert len(sub) == len(sample_sub)
sub.to_csv(OUTPUT_DIR / "submission.csv", index=False)
np.save(OUTPUT_DIR / "test_probs.npy", final_probs)
print("Saved →", OUTPUT_DIR / "submission.csv")
display(sub.head()); print(sub.iloc[:, 1].value_counts(normalize=True).round(3))

## ☁️ Colab: Submission download + Drive এ backup

In [ ]:
SAVE_TO_DRIVE = False                                  # True → outputs folder Google Drive এ copy হবে
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/datathon/outputs/audio"

if IN_COLAB:
    from google.colab import files, drive
    sub_path = OUTPUT_DIR / "submission.csv"
    if sub_path.exists():
        files.download(str(sub_path))                  # browser এ download হবে
    if SAVE_TO_DRIVE:
        drive.mount("/content/drive")
        shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT_DIR, dirs_exist_ok=True)
        print("Saved to Drive →", DRIVE_OUTPUT_DIR)
else:
    print("Local run — files আছে:", OUTPUT_DIR.resolve())

## 9. অন্য Audio task হলে?

| Task | কি করবো |
|---|---|
| **Speech-to-text (ASR)** | Whisper: `pipeline("automatic-speech-recognition", model="openai/whisper-small")` → `asr(path, generate_kwargs={"language": "bengali"})`; metric WER: `pip install jiwer` → `jiwer.wer(refs, hyps)` |
| **বাংলা ASR fine-tune** | `openai/whisper-small`/`medium` বা `facebook/wav2vec2-xls-r-300m` + CTC; text normalize (Unicode NFC) জরুরি |
| **Multi-label** (একই clip এ একাধিক sound) | `nn.BCEWithLogitsLoss()`, `sigmoid`, threshold tune; metric mAP / F1-micro |
| **Sound event detection / লম্বা recording** (BirdCLEF) | 5s chunk এ ভাগ → প্রতিটা chunk predict; train এ random crop; secondary label কে soft label |
| **Regression** (age, emotion intensity) | `num_classes=1`, `nn.MSELoss()`/`SmoothL1Loss()` |
| **Speaker / emotion / language ID** | wav2vec2 / WavLM / HuBERT embedding (Section 5) + LogReg — প্রায়ই যথেষ্ট |
| **Music** | `SR=22050`, chroma + tempo feature; `CLIP_SECONDS` বড় (5-10s) |

## ✅ Score বাড়ানোর checklist
1. `CLIP_SECONDS` আর `SR` data অনুযায়ী (duration plot দেখে) ঠিক করো
2. **SpecAugment + mixup + random crop** — audio তে overfitting কমানোর মূল অস্ত্র
3. বড় backbone (`efficientnet_b0 → b2`, `tf_efficientnetv2_s`), `N_MELS` 128 → 256
4. সব fold train → fold ensemble; CNN + AST embedding + handcrafted model blend
5. লম্বা test clip এ chunk-probability `mean` বনাম `max` দুটোই try করো
6. Class imbalance → `CrossEntropyLoss(weight=...)` বা oversampling